# Local Outlier Factor (LOF) - Complete Implementation

This notebook explains and implements LOF step by step, compares it with Isolation Forest, demonstrates hyperparameter tuning, and finishes with a real-world fraud detection example.

## Imports

### Explanation
This section explains **Imports** and demonstrates it with well-commented Python code. Run the cell below to observe the output and understand how LOF works.

In [ ]:
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Part 1: Create Data with Local & Global Outliers

### Explanation
This section explains **Part 1: Create Data with Local & Global Outliers** and demonstrates it with well-commented Python code. Run the cell below to observe the output and understand how LOF works.

In [ ]:
np.random.seed(42)

cluster1=np.random.randn(150,2)*0.3+[0,0]
cluster2=np.random.randn(30,2)*1.5+[3,3]
global_outlier=np.array([[6,6]])
local_outlier=np.array([[3,5]])

X=np.vstack([cluster1,cluster2,global_outlier,local_outlier])

scaler=StandardScaler()
X_scaled=scaler.fit_transform(X)

print("Dataset shape:",X.shape)

## Part 2: LOF vs Isolation Forest

### Explanation
This section explains **Part 2: LOF vs Isolation Forest** and demonstrates it with well-commented Python code. Run the cell below to observe the output and understand how LOF works.

In [ ]:
lof=LocalOutlierFactor(n_neighbors=20,contamination=0.05)
lof_labels=lof.fit_predict(X_scaled)
lof_scores=lof.negative_outlier_factor_

iso=IsolationForest(contamination=0.05,random_state=42)
iso_labels=iso.fit_predict(X_scaled)

print("LOF anomalies:",(lof_labels==-1).sum())
print("Isolation Forest anomalies:",(iso_labels==-1).sum())

fig,ax=plt.subplots(1,3,figsize=(18,5))
ax[0].scatter(X[:,0],X[:,1]); ax[0].set_title("Original")

ax[1].scatter(X[iso_labels==1,0],X[iso_labels==1,1],label="Normal")
ax[1].scatter(X[iso_labels==-1,0],X[iso_labels==-1,1],marker="x",s=80,label="Outlier")
ax[1].set_title("Isolation Forest"); ax[1].legend()

ax[2].scatter(X[lof_labels==1,0],X[lof_labels==1,1],label="Normal")
ax[2].scatter(X[lof_labels==-1,0],X[lof_labels==-1,1],marker="x",s=80,label="Outlier")
ax[2].set_title("LOF"); ax[2].legend()
plt.show()

## Part 3: Understanding LOF Scores

### Explanation
This section explains **Part 3: Understanding LOF Scores** and demonstrates it with well-commented Python code. Run the cell below to observe the output and understand how LOF works.

In [ ]:
indices=[0,150,180,181]
names=["Dense","Sparse","Global Outlier","Local Outlier"]

for i,n in zip(indices,names):
    print(n,":",round(-lof_scores[i],3),"Detected" if lof_labels[i]==-1 else "Normal")

print("\nInterpretation")
print("~1 : Normal")
print(">1.5 : Suspicious")
print(">2 : Strong Outlier")

## Part 4: Effect of n_neighbors

### Explanation
This section explains **Part 4: Effect of n_neighbors** and demonstrates it with well-commented Python code. Run the cell below to observe the output and understand how LOF works.

In [ ]:
k_values=[5,10,20,50]
fig,axes=plt.subplots(2,2,figsize=(12,10))
for idx,k in enumerate(k_values):
    model=LocalOutlierFactor(n_neighbors=k,contamination=0.05)
    labels=model.fit_predict(X_scaled)
    ax=axes[idx//2,idx%2]
    ax.scatter(X[labels==1,0],X[labels==1,1])
    ax.scatter(X[labels==-1,0],X[labels==-1,1],marker="x",s=80)
    ax.set_title(f"k={k}")
plt.tight_layout()
plt.show()

## Part 5: Credit Card Fraud Example

### Explanation
This section explains **Part 5: Credit Card Fraud Example** and demonstrates it with well-commented Python code. Run the cell below to observe the output and understand how LOF works.

In [ ]:
np.random.seed(42)

normal=pd.DataFrame({
"amount":np.concatenate([np.random.lognormal(3,0.5,200),np.random.lognormal(5,0.3,100)]),
"hour":np.random.normal(14,4,300)
})

fraud1=pd.DataFrame({
"amount":np.random.lognormal(3,0.3,15),
"hour":np.random.uniform(2,5,15)
})

fraud2=pd.DataFrame({
"amount":np.random.lognormal(8,0.2,5),
"hour":np.random.normal(14,2,5)
})

transactions=pd.concat([normal,fraud1,fraud2],ignore_index=True)

scaled=scaler.fit_transform(transactions)
lof=LocalOutlierFactor(n_neighbors=20,contamination=0.06)
labels=lof.fit_predict(scaled)

print("Transactions:",len(transactions))
print("Detected anomalies:",(labels==-1).sum())

plt.figure(figsize=(10,5))
plt.scatter(transactions.loc[labels==1,"amount"],transactions.loc[labels==1,"hour"],label="Normal")
plt.scatter(transactions.loc[labels==-1,"amount"],transactions.loc[labels==-1,"hour"],marker="x",s=80,label="Anomaly")
plt.xlabel("Amount")
plt.ylabel("Hour")
plt.legend()
plt.show()

# Summary

- Learned how LOF detects anomalies using local density.
- Compared LOF with Isolation Forest.
- Explored LOF scores.
- Studied the effect of `n_neighbors`.
- Applied LOF to a fraud detection example.
